# P5c: FigQuant dequantization gate

This notebook measures the actual gate: V3 must beat V1 on peak RSS and pass numerical correctness against the V1 FP32 reference. Each variant/iteration runs in a fresh process, with wall-clock timing and allocator reclamation recorded.

In [ ]:
REPO_URL='https://github.com/Harboria-Labs/littlefig.git'
REPO_BRANCH='research/p1-figmezo-verify'
REPO_DIR='/content/littlefig'
import os, subprocess, sys
if not os.path.exists(REPO_DIR): subprocess.run(['git','clone','--branch',REPO_BRANCH,REPO_URL,REPO_DIR],check=True)
os.chdir(REPO_DIR)

In [ ]:
!pip -q install psutil
!python -m pip check

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,'benchmark/experiment_dequant_variants_p5c.py','--iterations','3','--batch','2','--seq','256','--tile','128','--results-path','/content/p5c_results.json'],check=True)

In [ ]:
import json
r=json.load(open('/content/p5c_results.json'))
from collections import defaultdict
s=defaultdict(list)
for x in r['cases']: s[x['variant']].append(x)
for v, xs in s.items():
 print(v, 'peak RSS=',max(x['rss_peak_mib'] for x in xs), 'median ms=',sorted(x['wall_ms'] for x in xs)[len(xs)//2], 'max error=',max(x['max_abs_error'] for x in xs), 'correct=',all(x['correctness_pass'] for x in xs))
v1=max(x['rss_peak_mib'] for x in s['v1_fp32_full']); v3=max(x['rss_peak_mib'] for x in s['v3_bf16_tiled']); print('P5c memory gate:',v3 < v1, 'correctness gate:',all(x['correctness_pass'] for x in s['v3_bf16_tiled']))